In [ ]:
import yfinance as yf

ticker = yf.Ticker("^GSPC")


df = ticker.history(start="1990-01-01", auto_adjust=True)


df.drop(columns=['Dividends','Stock Splits'],inplace=True)


df

In [ ]:
import numpy as np
from sklearn.preprocessing import  MinMaxScaler

close_price  =  df['Close'].values


scaler_sp500 = MinMaxScaler(feature_range=(0,1))

scaled_df = scaler_sp500.fit_transform(close_price.reshape(-1,1))

train_size = int(len(scaled_df)*0.8)

train_data = scaled_df[:train_size]
test_data = scaled_df[train_size:]

In [ ]:
def seq(data,seq_len):

    x,y = [],[]

    for i in range(seq_len,len(data)):
        x.append(data[i-seq_len:i,0])
        y.append(data[i,0])
    return np.array(x), np.array(y)

In [ ]:
seq_len = 60

x_train,y_train = seq(train_data,seq_len)

x_test,y_test =  seq(test_data,seq_len)


x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers,models
from tensorflow.keras.layers import LSTM,GRU

lstm_sp500 = models.Sequential()

lstm_sp500.add(LSTM(units=100,return_sequences=True,input_shape=(x_train.shape[1],1)))
lstm_sp500.add(layers.Dropout(0.2))

lstm_sp500.add(LSTM(units=100,return_sequences=False))
lstm_sp500.add(layers.Dropout(0.2))

lstm_sp500.add(layers.Dense(1)) 

lstm_sp500.compile(optimizer='adam',loss='mean_squared_error',metrics=['mae'])

lstm_sp500.fit(x_train,y_train,epochs=100,batch_size=32)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers,models
from tensorflow.keras.layers import LSTM,GRU
gru_sp500 = models.Sequential()

gru_sp500.add(GRU(units=100, return_sequences=True, input_shape=(x_train.shape[1], 1)))
gru_sp500.add(layers.Dropout(0.2))

gru_sp500.add(GRU(units=100, return_sequences=False))
gru_sp500.add(layers.Dropout(0.2))

gru_sp500.add(layers.Dense(1))

gru_sp500.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
gru_sp500.fit(x_train, y_train, epochs=100, batch_size=32)

In [ ]:
# 🔍 Predict using LSTM on full test set
lstm_pred = lstm_sp500.predict(x_test)
y_test_actual = scaler_sp500.inverse_transform(y_test.reshape(-1, 1))
lstm_pred_actual = scaler_sp500.inverse_transform(lstm_pred)

# 🔮 Forecast next 10 days using LSTM
future_preds = []
current_seq = x_test[-1]

for _ in range(10):
    next_scaled = lstm_sp500.predict(current_seq.reshape(1, seq_len, 1))[0][0]
    future_preds.append(next_scaled)
    current_seq = np.append(current_seq[1:], [[next_scaled]], axis=0)

future_prices = scaler_sp500.inverse_transform(np.array(future_preds).reshape(-1, 1))

# 🧪 Combine last 60 predicted + next 10 forecasted
last_60_pred = lstm_pred_actual[-60:]  # Last 60 predicted values
extended_pred = np.concatenate((last_60_pred, future_prices), axis=0)

# 📊 Plot actual vs extended prediction
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# 🎨 Styling
sns.set(style="whitegrid")
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['lines.markersize'] = 6

plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), extended_pred, label='LSTM Prediction + Forecast (Next 10 Days)', color='orange', marker='o')
plt.title("📈 LSTM: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
# 🎨 Styling
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

sns.set(style="whitegrid")
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['lines.markersize'] = 6

# 🔍 Predict on test set
gru_pred = gru_sp500.predict(x_test)
gru_pred_actual = scaler_sp500.inverse_transform(gru_pred)
y_test_actual = scaler_sp500.inverse_transform(y_test.reshape(-1, 1))

# 🔮 Forecast next 10 days
gru_future_preds = []
gru_current_seq = x_test[-1]

for _ in range(10):
    next_scaled = gru_sp500.predict(gru_current_seq.reshape(1, x_test.shape[1], 1))[0][0]
    gru_future_preds.append(next_scaled)
    gru_current_seq = np.append(gru_current_seq[1:], [[next_scaled]], axis=0)

gru_forecast = scaler_sp500.inverse_transform(np.array(gru_future_preds).reshape(-1, 1))

# 🧠 Combine predictions
gru_last_60_pred = gru_pred_actual[-60:]
gru_extended_pred = np.concatenate([gru_last_60_pred, gru_forecast])

# 📊 Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU Prediction + Forecast (Next 10 Days)', color='green', marker='o')
plt.title("📈 GRU: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual', color='black', marker='o')
plt.plot(range(70), extended_pred, label='LSTM', color='orange', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU', color='green', marker='o')
plt.title("📊 LSTM vs GRU: 60-Day Prediction + 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
import joblib
joblib.dump(df, 'df_sp500.pkl')
joblib.dump(scaler_sp500, 'scaler_sp500.pkl')
gru_sp500.save("gru_model_sp500.keras")

In [ ]:
from sklearn.metrics import mean_absolute_error

print("LSTM MAE:", mean_absolute_error(y_test_actual, lstm_pred_actual))
print("GRU  MAE:", mean_absolute_error(y_test_actual, gru_pred_actual))

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler


ticker1 = yf.Ticker("AAPL")
df1 = ticker1.history(start="1990-01-01", auto_adjust=True)
close_price = df1['Close'].values
scaler_aapl = MinMaxScaler(feature_range=(0, 1))
scaled_df = scaler_aapl.fit_transform(close_price.reshape(-1, 1))

train_size = int(len(scaled_df) * 0.8)
train_data = scaled_df[:train_size]
test_data = scaled_df[train_size:]

def seq(data, seq_len):
    x, y = [], []
    for i in range(seq_len, len(data)):
        x.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(x), np.array(y)

seq_len = 60
x_train, y_train = seq(train_data, seq_len)
x_test, y_test = seq(test_data, seq_len)

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import LSTM, GRU

# LSTM Model
lstm_aapl = models.Sequential([
    LSTM(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    LSTM(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
lstm_aapl.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
lstm_aapl.fit(x_train, y_train, epochs=100, batch_size=32)

# GRU Model
gru_aapl = models.Sequential([
    GRU(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    GRU(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
gru_aapl.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
gru_aapl.fit(x_train, y_train, epochs=100, batch_size=32)

# LSTM Prediction + Forecast
lstm_pred = lstm_aapl.predict(x_test)
y_test_actual = scaler_aapl.inverse_transform(y_test.reshape(-1, 1))
lstm_pred_actual = scaler_aapl.inverse_transform(lstm_pred)

future_preds = []
current_seq = x_test[-1]
for _ in range(10):
    next_scaled = lstm_aapl.predict(current_seq.reshape(1, seq_len, 1))[0][0]
    future_preds.append(next_scaled)
    current_seq = np.append(current_seq[1:], [[next_scaled]], axis=0)

future_prices = scaler_aapl.inverse_transform(np.array(future_preds).reshape(-1, 1))
last_60_pred = lstm_pred_actual[-60:]
extended_pred = np.concatenate((last_60_pred, future_prices), axis=0)

# GRU Prediction + Forecast
gru_pred = gru_aapl.predict(x_test)
gru_pred_actual = scaler_aapl.inverse_transform(gru_pred)

gru_future_preds = []
gru_current_seq = x_test[-1]
for _ in range(10):
    next_scaled = gru_aapl.predict(gru_current_seq.reshape(1, seq_len, 1))[0][0]
    gru_future_preds.append(next_scaled)
    gru_current_seq = np.append(gru_current_seq[1:], [[next_scaled]], axis=0)

gru_forecast = scaler_aapl.inverse_transform(np.array(gru_future_preds).reshape(-1, 1))
gru_last_60_pred = gru_pred_actual[-60:]
gru_extended_pred = np.concatenate((gru_last_60_pred, gru_forecast), axis=0)

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl


sns.set(style="whitegrid")
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['lines.markersize'] = 6

# LSTM Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), extended_pred, label='LSTM Prediction + Forecast (Next 10 Days)', color='orange', marker='o')
plt.title("📈 AAPL LSTM: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

# GRU Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU Prediction + Forecast (Next 10 Days)', color='green', marker='o')
plt.title("📈 AAPL GRU: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()


# Combined Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual', color='black', marker='o')
plt.plot(range(70), extended_pred, label='LSTM', color='orange', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU', color='green', marker='o')
plt.title("📊 AAPL LSTM vs GRU: 60-Day Prediction + 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
joblib.dump(df1, 'df_aapl.pkl')
joblib.dump(scaler_aapl, 'scaler_aapl.pkl')
gru_aapl.save("gru_model_aapl.keras")

In [ ]:
from sklearn.metrics import mean_absolute_error

print("LSTM MAE:", mean_absolute_error(y_test_actual, lstm_pred_actual))
print("GRU  MAE:", mean_absolute_error(y_test_actual, gru_pred_actual))

In [ ]:
import yfinance as yf
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import LSTM, GRU
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# 🔹 Load TSLA data
ticker2 = yf.Ticker("TSLA")
df2 = ticker2.history(start="1990-01-01", auto_adjust=True)
close_price = df2['Close'].values

# 🔹 Scaling
scaler_tsla = MinMaxScaler(feature_range=(0, 1))
scaled_df = scaler_tsla.fit_transform(close_price.reshape(-1, 1))
train_size = int(len(scaled_df) * 0.8)
train_data = scaled_df[:train_size]
test_data = scaled_df[train_size:]

# 🔹 Sequence creation
def seq(data, seq_len):
    x, y = [], []
    for i in range(seq_len, len(data)):
        x.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(x), np.array(y)

seq_len = 60
x_train, y_train = seq(train_data, seq_len)
x_test, y_test = seq(test_data, seq_len)

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

# 🔹 LSTM Model
lstm_tsla = models.Sequential([
    LSTM(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    LSTM(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
lstm_tsla.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
lstm_tsla.fit(x_train, y_train, epochs=100, batch_size=32)

# 🔹 GRU Model
gru_tsla = models.Sequential([
    GRU(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    GRU(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
gru_tsla.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
gru_tsla.fit(x_train, y_train, epochs=100, batch_size=32)

# 🔹 LSTM Prediction + Forecast
lstm_pred = lstm_tsla.predict(x_test)
y_test_actual = scaler_tsla.inverse_transform(y_test.reshape(-1, 1))
lstm_pred_actual = scaler_tsla.inverse_transform(lstm_pred)

future_preds = []
current_seq = x_test[-1]
for _ in range(10):
    next_scaled = lstm_tsla.predict(current_seq.reshape(1, seq_len, 1))[0][0]
    future_preds.append(next_scaled)
    current_seq = np.append(current_seq[1:], [[next_scaled]], axis=0)

future_prices = scaler_tsla.inverse_transform(np.array(future_preds).reshape(-1, 1))
last_60_pred = lstm_pred_actual[-60:]
extended_pred = np.concatenate((last_60_pred, future_prices), axis=0)

# 🔹 GRU Prediction + Forecast
gru_pred = gru_tsla.predict(x_test)
gru_pred_actual = scaler_tsla.inverse_transform(gru_pred)

gru_future_preds = []
gru_current_seq = x_test[-1]
for _ in range(10):
    next_scaled = gru_tsla.predict(gru_current_seq.reshape(1, seq_len, 1))[0][0]
    gru_future_preds.append(next_scaled)
    gru_current_seq = np.append(gru_current_seq[1:], [[next_scaled]], axis=0)

gru_forecast = scaler_tsla.inverse_transform(np.array(gru_future_preds).reshape(-1, 1))
gru_last_60_pred = gru_pred_actual[-60:]
gru_extended_pred = np.concatenate((gru_last_60_pred, gru_forecast), axis=0)

# 🔹 Plotting
sns.set(style="whitegrid")
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['lines.markersize'] = 6

# LSTM Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), extended_pred, label='LSTM Prediction + Forecast (Next 10 Days)', color='orange', marker='o')
plt.title("📈 TSLA LSTM: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

# GRU Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU Prediction + Forecast (Next 10 Days)', color='green', marker='o')
plt.title("📈 TSLA GRU: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

# Combined Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual', color='black', marker='o')
plt.plot(range(70), extended_pred, label='LSTM', color='orange', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU', color='green', marker='o')
plt.title("📊 TSLA LSTM vs GRU: 60-Day Prediction + 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
joblib.dump(df2, 'df_tsla.pkl')
joblib.dump(scaler_tsla, 'scaler_tsla.pkl')
gru_tsla.save("gru_model_tsla.keras")

In [ ]:
from sklearn.metrics import mean_absolute_error

print("LSTM MAE:", mean_absolute_error(y_test_actual, lstm_pred_actual))
print("GRU  MAE:", mean_absolute_error(y_test_actual, gru_pred_actual))

In [ ]:
import yfinance as yf
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import LSTM, GRU
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# 🔹 Load RELIANCE data
ticker3 = yf.Ticker("RELIANCE.NS")
df3 = ticker3.history(start="1990-01-01", auto_adjust=True)
close_price = df3['Close'].values

# 🔹 Scaling
scaler_reliance = MinMaxScaler(feature_range=(0, 1))
scaled_df = scaler_reliance.fit_transform(close_price.reshape(-1, 1))
train_size = int(len(scaled_df) * 0.8)
train_data = scaled_df[:train_size]
test_data = scaled_df[train_size:]

# 🔹 Sequence creation
def seq(data, seq_len):
    x, y = [], []
    for i in range(seq_len, len(data)):
        x.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(x), np.array(y)

seq_len = 60
x_train, y_train = seq(train_data, seq_len)
x_test, y_test = seq(test_data, seq_len)

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

# 🔹 LSTM Model
lstm_reliance = models.Sequential([
    LSTM(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    LSTM(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
lstm_reliance.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
lstm_reliance.fit(x_train, y_train, epochs=100, batch_size=32)

# 🔹 GRU Model
gru_reliance = models.Sequential([
    GRU(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    GRU(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
gru_reliance.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
gru_reliance.fit(x_train, y_train, epochs=100, batch_size=32)

# 🔹 LSTM Prediction + Forecast
lstm_pred = lstm_reliance.predict(x_test)
y_test_actual = scaler_reliance.inverse_transform(y_test.reshape(-1, 1))
lstm_pred_actual = scaler_reliance.inverse_transform(lstm_pred)

future_preds = []
current_seq = x_test[-1]
for _ in range(10):
    next_scaled = lstm_reliance.predict(current_seq.reshape(1, seq_len, 1))[0][0]
    future_preds.append(next_scaled)
    current_seq = np.append(current_seq[1:], [[next_scaled]], axis=0)

future_prices = scaler_reliance.inverse_transform(np.array(future_preds).reshape(-1, 1))
last_60_pred = lstm_pred_actual[-60:]
extended_pred = np.concatenate((last_60_pred, future_prices), axis=0)

# 🔹 GRU Prediction + Forecast
gru_pred = gru_reliance.predict(x_test)
gru_pred_actual = scaler_reliance.inverse_transform(gru_pred)


gru_future_preds = []
gru_current_seq = x_test[-1]
for _ in range(10):
    next_scaled = gru_reliance.predict(gru_current_seq.reshape(1, seq_len, 1))[0][0]
    gru_future_preds.append(next_scaled)
    gru_current_seq = np.append(gru_current_seq[1:], [[next_scaled]], axis=0)

gru_forecast = scaler_reliance.inverse_transform(np.array(gru_future_preds).reshape(-1, 1))
gru_last_60_pred = gru_pred_actual[-60:]
gru_extended_pred = np.concatenate((gru_last_60_pred, gru_forecast), axis=0)

# 🔹 Plotting
sns.set(style="whitegrid")
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['lines.markersize'] = 6

# LSTM Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), extended_pred, label='LSTM Prediction + Forecast (Next 10 Days)', color='orange', marker='o')
plt.title("📈 RELIANCE LSTM: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()




# GRU Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU Prediction + Forecast (Next 10 Days)', color='green', marker='o')
plt.title("📈 RELIANCE GRU: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

# Combined Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual', color='black', marker='o')
plt.plot(range(70), extended_pred, label='LSTM', color='orange', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU', color='green', marker='o')
plt.title("📊 RELIANCE LSTM vs GRU: 60-Day Prediction + 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
joblib.dump(df3, 'df_reliance.pkl')
joblib.dump(scaler_reliance, 'scaler_reliance.pkl')
gru_reliance.save("gru_model_reliance.keras")

In [ ]:
from sklearn.metrics import mean_absolute_error

print("LSTM MAE:", mean_absolute_error(y_test_actual, lstm_pred_actual))
print("GRU  MAE:", mean_absolute_error(y_test_actual, gru_pred_actual))

In [ ]:
import yfinance as yf
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import LSTM, GRU
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

# 🔹 Load INFOSYS data
ticker4 = yf.Ticker("INFY.NS")
df4 = ticker4.history(start="1990-01-01", auto_adjust=True)
close_price = df4['Close'].values

# 🔹 Scaling
scaler_infosys = MinMaxScaler(feature_range=(0, 1))
scaled_df = scaler_infosys.fit_transform(close_price.reshape(-1, 1))
train_size = int(len(scaled_df) * 0.8)
train_data = scaled_df[:train_size]
test_data = scaled_df[train_size:]

# 🔹 Sequence creation
def seq(data, seq_len):
    x, y = [], []
    for i in range(seq_len, len(data)):
        x.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(x), np.array(y)

seq_len = 60
x_train, y_train = seq(train_data, seq_len)
x_test, y_test = seq(test_data, seq_len)

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

# 🔹 LSTM Model
lstm_infosys = models.Sequential([
    LSTM(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    LSTM(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
lstm_infosys.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
lstm_infosys.fit(x_train, y_train, epochs=100, batch_size=32)

# 🔹 GRU Model
gru_infosys = models.Sequential([
    GRU(100, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    layers.Dropout(0.2),
    GRU(100),
    layers.Dropout(0.2),
    layers.Dense(1)
])
gru_infosys.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
gru_infosys.fit(x_train, y_train, epochs=100, batch_size=32)

# 🔹 LSTM Prediction + Forecast
lstm_pred = lstm_infosys.predict(x_test)
y_test_actual = scaler_infosys.inverse_transform(y_test.reshape(-1, 1))
lstm_pred_actual = scaler_infosys.inverse_transform(lstm_pred)

future_preds = []
current_seq = x_test[-1]
for _ in range(10):
    next_scaled = lstm_infosys.predict(current_seq.reshape(1, seq_len, 1))[0][0]
    future_preds.append(next_scaled)
    current_seq = np.append(current_seq[1:], [[next_scaled]], axis=0)

future_prices = scaler_infosys.inverse_transform(np.array(future_preds).reshape(-1, 1))
last_60_pred = lstm_pred_actual[-60:]
extended_pred = np.concatenate((last_60_pred, future_prices), axis=0)

# 🔹 GRU Prediction + Forecast
gru_pred = gru_infosys.predict(x_test)
gru_pred_actual = scaler_infosys.inverse_transform(gru_pred)


gru_future_preds = []
gru_current_seq = x_test[-1]
for _ in range(10):
    next_scaled = gru_infosys.predict(gru_current_seq.reshape(1, seq_len, 1))[0][0]
    gru_future_preds.append(next_scaled)
    gru_current_seq = np.append(gru_current_seq[1:], [[next_scaled]], axis=0)

gru_forecast = scaler_infosys.inverse_transform(np.array(gru_future_preds).reshape(-1, 1))
gru_last_60_pred = gru_pred_actual[-60:]
gru_extended_pred = np.concatenate((gru_last_60_pred, gru_forecast), axis=0)

# 🔹 Plotting
sns.set(style="whitegrid")
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['lines.markersize'] = 6

# LSTM Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), extended_pred, label='LSTM Prediction + Forecast (Next 10 Days)', color='orange', marker='o')
plt.title("📈 INFOSYS LSTM: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

# GRU Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual (Last 60 Days)', color='blue', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU Prediction + Forecast (Next 10 Days)', color='green', marker='o')
plt.title("📈 INFOSYS GRU: Last 60-Day Prediction + Next 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()


# Combined Plot
plt.figure(figsize=(12, 6))
plt.plot(range(60), y_test_actual[-60:], label='Actual', color='black', marker='o')
plt.plot(range(70), extended_pred, label='LSTM', color='orange', marker='o')
plt.plot(range(70), gru_extended_pred, label='GRU', color='green', marker='o')
plt.title("📊 INFOSYS LSTM vs GRU: 60-Day Prediction + 10-Day Forecast", pad=20)
plt.xlabel("Day Index")
plt.ylabel("Close Price")
plt.legend(loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
joblib.dump(df4, 'df_infosys.pkl')
joblib.dump(scaler_infosys, 'scaler_infosys.pkl')
gru_infosys.save("gru_model_infosys.keras")

In [ ]:
from sklearn.metrics import mean_absolute_error

print("LSTM MAE:", mean_absolute_error(y_test_actual, lstm_pred_actual))
print("GRU  MAE:", mean_absolute_error(y_test_actual, gru_pred_actual))

In [ ]:
import streamlit
import tensorflow as tf
import sklearn
import yfinance
import matplotlib
import seaborn
import joblib
import pandas as pd
import numpy as np

print("📦 Streamlit:", streamlit.__version__)
print("📦 TensorFlow:", tf.__version__)
print("📦 scikit-learn:", sklearn.__version__)
print("📦 yfinance:", yfinance.__version__)
print("📦 Matplotlib:", matplotlib.__version__)
print("📦 Seaborn:", seaborn.__version__)
print("📦 Joblib:", joblib.__version__)
print("📦 Pandas:", pd.__version__)
print("📦 NumPy:", np.__version__)